# 03 - Structured output and image understanding

Goal: turn model output into validated Python data and provide an image as input. Allow 30 minutes.

Run cells from top to bottom. Complete notebook 00 first. Each notebook starts with fresh state.


In [ ]:
from google.genai import types
from course_config import PROJECT_ID, MODEL_ID, EMBEDDING_LOCATION, make_client
client = make_client()
print('Project:', PROJECT_ID, '| Model:', MODEL_ID)


In [ ]:
from pydantic import BaseModel, Field
class Ticket(BaseModel):
    category: str = Field(description='One of access, billing, or technical')
    summary: str
    urgent: bool

result = client.models.generate_content(model=MODEL_ID,
    contents='Classify this support ticket: My company laptop was stolen this morning.',
    config=types.GenerateContentConfig(response_mime_type='application/json',
        response_schema=Ticket, max_output_tokens=1200))
ticket = Ticket.model_validate_json(result.text)
print(ticket.model_dump())


Schema validation checks structure, not factual truth. A string description does not enforce an enum. Exercise: replace category with Literal values to enforce the set.


In [ ]:
from PIL import Image, ImageDraw
from io import BytesIO
from IPython.display import display
picture = Image.new('RGB', (640, 280), 'white')
draw = ImageDraw.Draw(picture)
draw.text((30, 20), 'Fictional quarterly orders', fill='black')
for i, (label, value) in enumerate([('Q1', 40), ('Q2', 75), ('Q3', 55)]):
    y = 60 + i * 65
    draw.text((20, y + 10), label, fill='black')
    draw.rectangle((65, y, 65 + value * 6, y + 35), fill='steelblue')
    draw.text((75 + value * 6, y + 10), str(value), fill='black')
display(picture)
buffer = BytesIO()
picture.save(buffer, format='PNG')
response = client.models.generate_content(model=MODEL_ID,
    contents=['Which quarter has the most orders? State the values shown.',
              types.Part.from_bytes(data=buffer.getvalue(), mime_type='image/png')],
    config=types.GenerateContentConfig(max_output_tokens=1200))
print(response.text)


## Try it yourself

Enforce the ticket category with typing.Literal. Add a fourth bar to the image and check the model answer against the underlying numbers.
